# V6.5-21 — Dữ liệu v6: đào ảnh khó bằng teacher v5 (phần A) → negative khó / diffusion (phần B, C)

**Vì sao:** hệ 2 tầng đạt MEAN_5 0.828 trên val_clean (CI95 0.798–0.859, `data/label_audit/preflight/system_two_tier*.py`),
trần là teacher. Class yếu nhất là **FO** (train chỉ 955 box, P = 0.55: báo nhầm nhiều) và **flashover** (1,245 box,
student kém teacher 6 điểm). Phần A tìm *chính xác* model sai ở đâu trước khi sinh thêm dữ liệu.

**Phần A (notebook này, ~10 phút GPU trên 4060):** teacher v5 chạy trên **toàn bộ train** (ảnh vuông 640, như khi đo) →
- **GT bị bỏ sót**: box nhãn mà teacher không thấy (conf < `MISS_CONF`), theo class / kích thước vật / ánh sáng;
- **báo nhầm**: box teacher conf ≥ `FP_CONF` không khớp nhãn → *nhầm class khác* / *lệch box* / *nền* (nền = negative khó **hoặc thiếu nhãn**, phải xem tay);
- CSV + **contact sheet** cho FO và flashover để xem bằng mắt.

**Đọc kết quả cho đúng:** teacher *đã train trên chính tập này*, nên trên train nó sai ít hơn trên ảnh mới. Một GT mà teacher
**vẫn** bỏ sót trên train gần như chắc là **rất khó hoặc nhãn sai**; một box báo nhầm conf cao trên train hay là **thiếu nhãn**.
Cả hai đều phải xem contact sheet trước khi dùng.

**Cách dùng:** giống notebook 20 (tự tải dataset `v6p5_v5` + teacher v5 từ Camber nếu máy chưa có). **Run All**.
Thử nhanh: `DRY_RUN = True` (300 ảnh). Chạy lại thì cell 4 chạy tiếp từ chỗ dừng.

In [ ]:
# 1. CẤU HÌNH — mặc định chạy được luôn
import os, sys, platform, shutil, time, json, random
from pathlib import Path

RUN_NAME = "v6p5_v6_data"             # thư mục artifacts/<RUN_NAME>
DATASET_NAME = "v6p5_v5"
STASH_DATASET = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_v5"
STASH_INIT = "stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_v5_20261013"   # teacher/best.pt
SPLIT = "train"                       # tập để đào (KHÔNG đào val/val_clean để chọn dữ liệu: sẽ làm số đo lạc quan)
IMGSZ = 640                           # ảnh vuông như Pi / như lúc đo hệ 2 tầng
MISS_CONF = 0.25                      # GT chỉ được teacher thấy với conf < ngưỡng này = bỏ sót
FP_CONF = 0.5                         # box teacher conf ≥ ngưỡng này mà không khớp nhãn = báo nhầm
FOCUS = ["foreign_object", "insulator_flashover"]   # class được làm contact sheet
SHEET_N = 48                          # số ô mỗi contact sheet
DARK_LUM, BRIGHT_LUM, BACKLIT = 60, 200, 60        # ánh sáng của VẬT (trong box nhãn, 0–255): tối < 60, chói > 200,
                                                   # ngược sáng = vật tối hơn cả ảnh > 60 (ảnh nền trời/tuyết luôn nhiều pixel trắng nên không đo theo cả ảnh)
DRY_RUN = os.environ.get("DRY_RUN", "0") == "1"     # hoặc đặt True: chỉ 300 ảnh

# ---- tự động: đường dẫn, phần cứng (không cần sửa) ----
import torch, ultralytics
from ultralytics import YOLO
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
for line in (REPO / ".env").read_text().splitlines() if (REPO / ".env").is_file() else []:   # CAMBER_API_KEY... (không in ra)
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
DATA_DIR = Path(os.environ.get("V6P5_DATA_DIR", REPO / "datasets"))
OUT = REPO / "artifacts" / (RUN_NAME + ("_dryrun" if DRY_RUN else ""))
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand", "person"]
DEFECTS = NAMES[1:6]
WINDOWS = platform.system() == "Windows"
CAMBER = os.environ.get("CAMBER_BIN") or shutil.which("camber") or str(Path.home() / ".camber/bin" / ("camber.exe" if WINDOWS else "camber"))
if ultralytics.__version__ != "8.4.172":
    print(f"CẢNH BÁO: dự án dùng ultralytics==8.4.172, đang có {ultralytics.__version__} (số có thể lệch chút ít)")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
BATCH = 32 if DEVICE == 0 else 4
DATA_DIR.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "progress.log"            # tiến độ cell 4 (mở file này để xem khi chạy bằng nbconvert)
print(f"{platform.system()} | {torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'} | ultralytics {ultralytics.__version__} | DRY_RUN {DRY_RUN}")
print("data:", DATA_DIR, "| kết quả:", OUT)

In [ ]:
# 2. TẢI DATASET từ Camber (bỏ qua nếu đã có) — giống notebook 20
import hashlib, subprocess, tarfile, random, yaml

def camber_cp(src, dst):
    if not (Path(CAMBER).exists() or shutil.which(CAMBER)):
        raise RuntimeError(f"Không thấy Camber CLI ({CAMBER}). Cài CLI hoặc đặt CAMBER_BIN trong .env; "
                           f"hoặc tự tải {src} về {dst}")
    subprocess.run([CAMBER, "stash", "cp", src, str(dst)], check=True, stdout=subprocess.DEVNULL)

DS = DATA_DIR / DATASET_NAME
if not (DS / "manifest.json").is_file():
    tar = DATA_DIR / f"{DATASET_NAME}.tar"
    for f in (f"{DATASET_NAME}.tar.sha256", f"{DATASET_NAME}.package.json", f"{DATASET_NAME}.tar"):
        if not (DATA_DIR / f).is_file():
            print("tải", f, "..."); camber_cp(f"{STASH_DATASET}/{f}", DATA_DIR / f)
    want = (DATA_DIR / f"{DATASET_NAME}.tar.sha256").read_text().split()[0]
    h = hashlib.sha256()
    with tar.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != want:
        tar.unlink(); raise RuntimeError("sha256 không khớp (tải lỗi) → đã xoá file tar, chạy lại cell này")
    print("giải nén ..."); tarfile.open(tar).extractall(DATA_DIR)
    tar.unlink()  # giữ ổ trống cho cache
for y, v in [("data.yaml", "images/val"), ("data_val_clean.yaml", "images/val_clean")]:   # path trong tar là của máy build
    cfg = yaml.safe_load((DS / y).read_text()); cfg.update(path=str(DS), train="images/train", val=v)
    (DS / y).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
assert [yaml.safe_load((DS / "data.yaml").read_text())["names"][i] for i in range(7)] == NAMES
pkg = json.loads((DATA_DIR / f"{DATASET_NAME}.package.json").read_text())
print("dataset OK:", DS, "| files:", pkg["files"], "| labels_sha256:", pkg["labels_sha256"][:16])


IMGS = sorted((DS / "images" / SPLIT).glob("*.jpg"))
if DRY_RUN:
    random.seed(0); IMGS = sorted(random.sample(IMGS, 300))
print(f"{SPLIT}: {len(IMGS)} ảnh")

In [ ]:
# 3. TEACHER v5 — dùng bản đã có trên máy (notebook 20 tải về) nếu có, không thì tải từ Camber
cands = [REPO / "artifacts/v6p5_v5_students/init/teacher_v5.pt", OUT / "init/teacher_v5.pt"]
TEACHER = next((p for p in cands if p.is_file()), cands[-1])
if not TEACHER.is_file():
    TEACHER.parent.mkdir(parents=True, exist_ok=True); print("tải teacher"); camber_cp(f"{STASH_INIT}/teacher/best.pt", TEACHER)
assert list(YOLO(str(TEACHER)).names.values()) == NAMES
print("teacher:", TEACHER)

In [ ]:
# 4. TEACHER CHẠY TRÊN TRAIN + đo ánh sáng từng ảnh → pred_<split>_<imgsz>.jsonl (chạy lại thì chạy tiếp)
import numpy as np, cv2

def log(msg):
    print(msg, flush=True)
    with LOG.open("a", encoding="utf-8") as f: f.write(f"{time.strftime('%H:%M:%S')} {msg}\n")

def light(img, lab):  # BGR + file nhãn → (độ sáng TB, std) của cả ảnh và của từng box nhãn (cùng thứ tự dòng), trên ảnh xám rộng 320
    g = cv2.cvtColor(cv2.resize(img, (320, max(1, round(320 * img.shape[0] / img.shape[1])))), cv2.COLOR_BGR2GRAY).astype(np.float32)
    H, W = g.shape; boxes = []
    for l in (lab.read_text().splitlines() if lab.exists() else []):
        if not l.strip(): continue
        _, x, y, w, h = map(float, l.split())
        x1, y1 = max(int((x - w / 2) * W), 0), max(int((y - h / 2) * H), 0)
        r = g[y1:max(int((y + h / 2) * H), y1 + 1), x1:max(int((x + w / 2) * W), x1 + 1)]
        boxes.append([round(float(r.mean()), 1), round(float(r.std()), 1)] if r.size else [-1, -1])
    return float(g.mean()), float(g.std()), boxes

PRED = OUT / f"pred_{SPLIT}_{IMGSZ}.jsonl"
done = {json.loads(l)["img"] for l in PRED.open(encoding="utf-8")} if PRED.exists() else set()
todo = [p for p in IMGS if p.name not in done]
log(f"bắt đầu: {len(done)} ảnh đã có, còn {len(todo)} | batch {BATCH} | {DEVICE}")
model, t0 = YOLO(str(TEACHER)), time.time()
with PRED.open("a", encoding="utf-8") as fh:
    for k in range(0, len(todo), BATCH):
        chunk = todo[k:k + BATCH]
        res = model.predict([str(p) for p in chunk], imgsz=IMGSZ, conf=0.01, max_det=100, rect=False, verbose=False,
                            device=DEVICE, batch=BATCH)
        for p, r in zip(chunk, res):
            lum, std, gl = light(r.orig_img, DS / "labels" / SPLIT / (p.stem + ".txt"))
            fh.write(json.dumps({"img": p.name, "w": r.orig_shape[1], "h": r.orig_shape[0], "lum": round(lum, 1),
                                 "std": round(std, 1), "gt_light": gl,
                                 "boxes": [[int(c), round(s, 4), *[round(v, 5) for v in b]] for c, s, b in
                                           zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxyn.tolist())]}) + "\n")
        n = k + len(chunk)
        if n % (BATCH * 20) < BATCH or n == len(todo):
            el = time.time() - t0; log(f"{n}/{len(todo)} ảnh | {n / el:.0f} ảnh/s | còn ~{(len(todo) - n) / max(n / el, 1e-9) / 60:.1f} phút")
del model; torch.cuda.empty_cache()
PREDS = {d["img"]: d for d in map(json.loads, PRED.open(encoding="utf-8")) if d["img"] in {p.name for p in IMGS}}
log(f"xong: {len(PREDS)} ảnh")

In [ ]:
# 5. GHÉP BOX TEACHER ↔ NHÃN: mỗi GT lấy conf của box cùng class khớp tốt nhất (IoU ≥ 0.5, ghép tham lam theo conf như mAP);
#    box teacher conf ≥ FP_CONF không khớp GT nào cùng class = báo nhầm, phân loại theo thứ nằm dưới nó
import pandas as pd

def iou(a, b):
    wh = np.clip(np.minimum(a[:, None, 2:], b[None, :, 2:]) - np.maximum(a[:, None, :2], b[None, :, :2]), 0, None)
    i = wh[..., 0] * wh[..., 1]; ar = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return i / (ar(a)[:, None] + ar(b)[None] - i + 1e-9)

def light_tag(obj, img):  # độ sáng của vật (trong box) so với ngưỡng và so với cả ảnh
    return "tối" if obj < DARK_LUM else "chói" if obj > BRIGHT_LUM else "ngược sáng" if img - obj > BACKLIT else "thường"

def size_tag(a):  # diện tích box / diện tích ảnh; ở 640: nhỏ ≈ cạnh < 64 px
    return "nhỏ" if a < 0.01 else "vừa" if a < 0.05 else "lớn"

GTS, FPS = [], []
for name, d in PREDS.items():
    lab = DS / "labels" / SPLIT / (Path(name).stem + ".txt")
    g = np.array([l.split() for l in lab.read_text().splitlines() if l.strip()] if lab.exists() else [], float).reshape(-1, 5)
    g = np.c_[g[:, :1], g[:, 1:3] - g[:, 3:5] / 2, g[:, 1:3] + g[:, 3:5] / 2]          # cls, x1, y1, x2, y2 (chuẩn hoá)
    gl = np.array(d["gt_light"], float).reshape(-1, 2); assert len(gl) == len(g), name
    P = np.array(d["boxes"], float).reshape(-1, 6)
    base = dict(img=name, img_lum=d["lum"])
    for c in range(1, 7):
        gc, glc, pc = g[g[:, 0] == c, 1:], gl[g[:, 0] == c], P[P[:, 0] == c]; pc = pc[np.argsort(-pc[:, 1])]
        best, used, hit = np.zeros(len(gc)), np.zeros(len(gc), bool), np.zeros(len(pc), bool)
        if len(gc) and len(pc):
            U = iou(pc[:, 2:], gc)
            for i in range(len(pc)):
                u = U[i].copy(); u[used] = -1; j = u.argmax()
                if u[j] >= 0.5: used[j], best[j], hit[i] = True, pc[i, 1], True
        for j, b in enumerate(gc):
            a = (b[2] - b[0]) * (b[3] - b[1])
            GTS.append({**base, "cls": NAMES[c], "conf": best[j], "area": a, "size": size_tag(a), "obj_lum": glc[j, 0],
                        "obj_std": glc[j, 1], "light": light_tag(glc[j, 0], d["lum"]), "box": [round(v, 5) for v in b]})
        for i in np.where(~hit & (pc[:, 1] >= FP_CONF))[0]:
            b = pc[i, 2:]; other = g[g[:, 0] != c]; same = gc
            uo = iou(b[None], other[:, 1:])[0] if len(other) else np.zeros(0)
            us = iou(b[None], same)[0] if len(same) else np.zeros(0)
            if len(uo) and uo.max() >= 0.5: kind = "nhầm " + NAMES[int(other[uo.argmax(), 0])]
            elif len(us) and us.max() >= 0.1: kind = "lệch box"
            else: kind = "nền / thiếu nhãn?"
            a = (b[2] - b[0]) * (b[3] - b[1])
            FPS.append({**base, "cls": NAMES[c], "conf": pc[i, 1], "kind": kind, "area": a, "size": size_tag(a), "box": [round(v, 5) for v in b]})
GT, FP = pd.DataFrame(GTS), pd.DataFrame(FPS)
GT["missed"] = GT.conf < MISS_CONF
print(f"{len(GT)} GT (class 1–6), {GT.missed.sum()} bị bỏ sót (conf < {MISS_CONF}); {len(FP)} box báo nhầm (conf ≥ {FP_CONF})")

In [ ]:
# 6. THỐNG KÊ: bỏ sót theo class / kích thước / ánh sáng; báo nhầm theo loại
from IPython.display import display
order = DEFECTS + ["person"]
pct = lambda s: f"{100 * s.mean():.1f}%"
t = GT.groupby("cls").agg(GT=("missed", "size"), bỏ_sót=("missed", "sum"), tỉ_lệ=("missed", pct)).reindex(order)
print("=== GT bị bỏ sót theo class ==="); display(t)
COLS = {"size": ["nhỏ", "vừa", "lớn"], "light": ["tối", "ngược sáng", "thường", "chói"]}
for by in ["size", "light"]:
    t = GT.pivot_table(index="cls", columns=by, values="missed", aggfunc=["mean", "size"]).reindex(order)
    t = t["mean"].reindex(columns=COLS[by]).map(lambda v: f"{100 * v:.1f}%" if v == v else "–") + " / " + t["size"].reindex(columns=COLS[by]).fillna(0).astype(int).astype(str)
    print(f"\n=== tỉ lệ bỏ sót / số GT theo {'kích thước vật' if by == 'size' else 'ánh sáng của vật'} ==="); display(t)
if len(FP):
    print(f"\n=== box báo nhầm (conf ≥ {FP_CONF}) theo loại ===")
    display(FP.pivot_table(index="cls", columns="kind", values="conf", aggfunc="size", fill_value=0).reindex(order).fillna(0).astype(int))

In [ ]:
# 7. XUẤT CSV: GT bỏ sót, box báo nhầm, và danh sách ảnh khó (điểm = số GT bỏ sót + số báo nhầm, chỉ class lỗi + person)
miss = GT[GT.missed].sort_values(["cls", "conf"]); miss.to_csv(OUT / "hard_missed_gt.csv", index=False)
FP.sort_values(["cls", "conf"], ascending=[True, False]).to_csv(OUT / "hard_false_pos.csv", index=False)
hard = pd.concat([miss.assign(t="missed"), FP.assign(t="fp")]).groupby("img").agg(
    missed=("t", lambda s: (s == "missed").sum()), fp=("t", lambda s: (s == "fp").sum()),
    classes=("cls", lambda s: ",".join(sorted(set(s)))), img_lum=("img_lum", "first")).assign(score=lambda d: d.missed + d.fp)
hard.sort_values("score", ascending=False).to_csv(OUT / "hard_images.csv")
print(f"{len(hard)} ảnh khó → {OUT / 'hard_images.csv'}; chi tiết: hard_missed_gt.csv ({len(miss)}), hard_false_pos.csv ({len(FP)})")
display(hard.sort_values("score", ascending=False).head(15))

In [ ]:
# 8. CONTACT SHEET cho FOCUS: (a) GT bị bỏ sót — khung xanh lá = nhãn, đỏ = box teacher cùng class gần nhất (nếu có);
#    (b) báo nhầm conf cao nhất — khung đỏ = box teacher, vàng = nhãn class khác nằm dưới. Mỗi ô: vùng quanh box ×3.
from IPython.display import Image as IPImage
SHEETS = OUT / "contact"; SHEETS.mkdir(exist_ok=True)

def tile(name, box, color, extra=(), text="", size=200):
    img = cv2.imread(str(DS / "images" / SPLIT / name)); H, W = img.shape[:2]
    x1, y1, x2, y2 = box[0] * W, box[1] * H, box[2] * W, box[3] * H
    cx, cy, s = (x1 + x2) / 2, (y1 + y2) / 2, max(x2 - x1, y2 - y1, 0.04 * max(W, H)) * 1.5
    X1, Y1, X2, Y2 = int(max(cx - s, 0)), int(max(cy - s, 0)), int(min(cx + s, W)), int(min(cy + s, H))
    for b, col in [*extra, (box, color)]:
        cv2.rectangle(img, (int(b[0] * W), int(b[1] * H)), (int(b[2] * W), int(b[3] * H)), col, max(2, W // 400))
    crop = img[Y1:Y2, X1:X2]; f = size / max(crop.shape[:2])
    crop = cv2.resize(crop, (max(1, int(crop.shape[1] * f)), max(1, int(crop.shape[0] * f))))
    out = np.full((size + 16, size, 3), 40, np.uint8); out[:crop.shape[0], :crop.shape[1]] = crop
    cv2.putText(out, text[:30], (2, size + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (255, 255, 255), 1, cv2.LINE_AA)
    return out

def sheet(tiles, path, cols=8):
    if not tiles: return None
    tiles += [np.full_like(tiles[0], 40)] * (-len(tiles) % cols)
    cv2.imwrite(str(path), np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])); return path

GREEN, RED = (0, 200, 0), (0, 0, 255)
ASCII = {"nhỏ": "small", "vừa": "mid", "lớn": "large", "tối": "dark", "ngược sáng": "backlit", "thường": "normal", "chói": "bright",
         "lệch box": "loc", "nền / thiếu nhãn?": "bg/unlabeled?"}   # cv2.putText không có dấu tiếng Việt
asc = lambda k: ASCII[k] if k in ASCII else str(k).replace("nhầm ", "as ")
for c in FOCUS:
    m = miss[miss.cls == c].sample(min(SHEET_N, (miss.cls == c).sum()), random_state=0)   # ngẫu nhiên = đại diện
    tl = []
    for _, r in m.iterrows():
        P = np.array(PREDS[r.img]["boxes"], float).reshape(-1, 6); P = P[P[:, 0] == NAMES.index(c)]
        ex = []
        if len(P):
            u = iou(np.array([r.box]), P[:, 2:])[0]
            if u.max() > 0: ex = [(P[u.argmax(), 2:], RED)]
        tl.append(tile(r.img, r.box, GREEN, ex, f"{r.conf:.2f} {asc(r['size'])} {asc(r.light)} {Path(r.img).stem[:10]}"))
    p1 = sheet(tl, SHEETS / f"{c}_missed.jpg")
    f = FP[FP.cls == c].sort_values("conf", ascending=False).head(SHEET_N); tl = []
    for _, r in f.iterrows():
        tl.append(tile(r.img, r.box, RED, [], f"{r.conf:.2f} {asc(r.kind)[:16]} {Path(r.img).stem[:10]}"))
    p2 = sheet(tl, SHEETS / f"{c}_false_pos.jpg")
    for p, title in [(p1, "GT bị bỏ sót (xanh = nhãn, đỏ = teacher)"), (p2, "báo nhầm conf cao nhất (đỏ = teacher)")]:
        if p: print(f"\n=== {c}: {title} → {p}"); display(IPImage(filename=str(p), width=1200))

### Đọc kết quả → quyết định phần B, C
- **Bỏ sót tập trung ở vật nhỏ** → ưu tiên tile / crop phóng to khi train, inpaint FO *nhỏ* ở phần C.
- **Bỏ sót tập trung ở vật tối / ngược sáng / chói** → phần C làm biến thể ánh sáng (img2img / ControlNet) cho đúng nhóm ảnh này.
- **Báo nhầm "nền / thiếu nhãn?"**: xem contact sheet — nếu là FO thật chưa gán nhãn → **sửa nhãn** (rule mới trong
  `scripts/label_audit_build.py`); nếu là vật giống FO (tạ chống rung, mối nối…) → **negative khó** cho phần B.
- **Báo nhầm "nhầm <class>"** → hai class đang lẫn nhau; xem nhãn có nhất quán không.
- GT bị bỏ sót mà contact sheet cho thấy **nhãn sai** → sửa nhãn, không đưa vào diffusion.

Phần B (negative khó) và C (diffusion) sẽ được thêm vào notebook này sau khi xem kết quả phần A.